Lanette Tyler   
ST554   
Spring 2026   
Project 3

# Purpose   
The purpose of this project is to use spark to fit a machine learning model and to handle streaming data. Pyspark's MLlib module will be used to fit the model. Streaming data to handle will be generated through a .py file run in the console. The data will be read, transformed (in part trhouhg making predictions using the model), and written to the console. Work will be documented in a github repo, and results will be demosntrated in a Zoom viedo.  

# Fitting the Model   
An elastic net model is fit to the provided data using cross validation. Data is read in, transformed, and a model is fit.

## Preliminary Steps: Import Modules and Functions, Create Spark Session

In [1]:
#import module 
import pandas as pd
import numpy as np

In [2]:
#import functions
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, PCA, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, CrossValidatorModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.types import StructType
from pyspark.sql.functions import col

In [3]:
#create spark session
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 06:28:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Read in Data

In [4]:
#read in the project data to a standard pandas data frame and view it
power_ml_data = pd.read_csv("https://www4.stat.ncsu.edu/online/datasets/power_ml_data.csv")
power_ml_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [5]:
power_ml_data.shape #check data set size

(47174, 10)

There are 47,174 observations in the data set.

In [6]:
#convert to a spark SQL data frame
power_ml_data = spark.createDataFrame(power_ml_data)
power_ml_data.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

## Transform Data

### Hour Column to Double Type, Response Variable Power_Zone_3 Renamed 'label'

In [7]:
#check data type of Hour column
power_ml_data

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, Power_Zone_3: double, Month: bigint, Hour: bigint]

The Hour column is read in as data type "bigint." It needs to be changed to "double" type.

In [8]:
#transform data type of Hour column to "double" type
SQLtrans = SQLTransformer(
                statement = """
                    SELECT Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, Diffuse_Flows,
                           Power_Zone_1, Power_Zone_2, Power_Zone_3 AS label, Month, CAST(HOUR AS DOUBLE) AS Hour
                    FROM __THIS__
                    """)

In [9]:
#view transformation results and object
SQLtrans.transform(power_ml_data).show(10)
SQLtrans.transform(power_ml_data)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1| 0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422|18899.27711|    1| 0.0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043|18442.40964|    1| 0.0|
|      5.853|    76.9|     0.081|               

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, label: double, Month: bigint, Hour: double]

The column Hour is now cast to double data type.

### Hour Column to Binary

In [10]:
#make Hour column binary, Hour < 6.5
binaryTrans = Binarizer(threshold = 6.5, inputCols = ["Hour"], 
                        outputCols = ["Daytime"])
binaryTrans.transform(
    SQLtrans.transform(power_ml_data)).show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1| 0.0|    0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422|18899.27711|    1| 0.0|    0.0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043|18442.40964|    

### Month Column One-Hot Encoded

In [11]:
#define One Hot Encoder trnasformer, and view object
OHEtrans = OneHotEncoder(inputCol = "Month", outputCol = "MonthVec")
OHEtrans

OneHotEncoder_1b625cdf42a1

In [12]:
#fit One hot Encoder Model using transformer, view model object
OHEmodel = OHEtrans.fit(
                binaryTrans.transform(
                    SQLtrans.transform(power_ml_data)))
OHEmodel

OneHotEncoderModel: uid=OneHotEncoder_1b625cdf42a1, dropLast=true, handleInvalid=error

In [13]:
#tansform data with model fit plus previous transformations, and view the object
OHEmodel.transform(
            binaryTrans.transform(
                SQLtrans.transform(power_ml_data)))

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, label: double, Month: bigint, Hour: double, Daytime: double, MonthVec: vector]

In [14]:
#apply transformations to the data set and view results
OHEmodel.transform(
            binaryTrans.transform(
                SQLtrans.transform(power_ml_data))) \
    .show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|      MonthVec|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|(12,[1],[1.0])|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|(12,[1],[1.0])|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1| 0.0|    0.0|(12,[1],[1.0])|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422|18899.27711|    1| 0.0|    0.0|(12,[1],[1.0])|
|     

### PCA Fit

In [15]:
#define assembler to create features column for principal components analysis
PCAassembler = VectorAssembler(inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], 
                            outputCol = "PCAfeatures", handleInvalid = "keep")

In [16]:
#tranform data for PCA using PCA assembler
PCAassembler.transform(
            OHEmodel.transform(
                    binaryTrans.transform(
                        SQLtrans.transform(power_ml_data)))) \
    .show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|      MonthVec|         PCAfeatures|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1| 0.0|    0.0|(12,[1],[1.0])|[6.313,74.5,0.08,...|
|      6.121|    75.0|

In [17]:
#create PCA transformer
PCAtrans = PCA(k = 2, inputCol = "PCAfeatures", outputCol = "PCAresult")
PCAtrans

PCA_0c1a418dc8ff

In [18]:
#Use PCA transformer to fit "model" to data
PCAmodel = PCAtrans.fit(
                PCAassembler.transform(
                    OHEmodel.transform(
                        binaryTrans.transform(
                            SQLtrans.transform(power_ml_data)))))
PCAmodel

PCAModel: uid=PCA_0c1a418dc8ff, k=2

In [19]:
#use "model" to transorm data
PCAmodel.transform(
                PCAassembler.transform(
                    OHEmodel.transform(
                        binaryTrans.transform(
                            SQLtrans.transform(power_ml_data))))) \
    .show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|      MonthVec|         PCAfeatures|           PCAresult|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|[1.79440486365695...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|[1.80604083009822...|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.1012

### Final Features Column for Transformed Data

In [20]:
#create features column for final assembly
assembler = VectorAssembler(inputCols = ["PCAresult", "Daytime", "Power_Zone_1", "Power_Zone_2", "MonthVec"], 
                            outputCol = "features", handleInvalid = "keep")
assembler

VectorAssembler_f5d1a394edbd

In [21]:
#view results
assembler.transform(
        PCAmodel.transform(
                PCAassembler.transform(
                    OHEmodel.transform(
                        binaryTrans.transform(
                            SQLtrans.transform(power_ml_data)))))) \
    .select("features").show(10, truncate = False)

+-------------------------------------------------------------------------------------+
|features                                                                             |
+-------------------------------------------------------------------------------------+
|(17,[0,1,3,4,6],[1.7944048636569518,-0.7412746447404432,34055.6962,16128.87538,1.0]) |
|(17,[0,1,3,4,6],[1.806040830098229,-0.7108534239558458,29814.68354,19375.07599,1.0]) |
|(17,[0,1,3,4,6],[1.8102297630563877,-0.7283113191158915,29128.10127,19006.68693,1.0])|
|(17,[0,1,3,4,6],[1.7986676517408817,-0.7220913032199923,28228.86076,18361.09422,1.0])|
|(17,[0,1,3,4,6],[1.8632872016379685,-0.7323046647776538,27335.6962,17872.34043,1.0]) |
|(17,[0,1,3,4,6],[1.878206745004608,-0.7628199906979571,26624.81013,17416.41337,1.0]) |
|(17,[0,1,3,4,6],[1.9152929871795528,-0.7636928919816393,25998.98734,16993.31307,1.0])|
|(17,[0,1,3,4,6],[1.9240054080702886,-0.7645388021423767,25446.07595,16661.39818,1.0])|
|(17,[0,1,3,4,6],[1.895019303530

### Pipeline   
All of the transformation steps are placed in a pipeline.

In [22]:
#define pipeline for transforming data, view pipeline object
pipeline = Pipeline(stages = [SQLtrans, binaryTrans, OHEtrans, PCAassembler, PCAtrans, assembler])
pipeline

Pipeline_dfee41365906

In [23]:
#make pipeline "model", view pipeline model object
pModel = pipeline.fit(power_ml_data)
pModel

PipelineModel_503e1c052a2f

In [24]:
#transform data using pipeline model, view transformed data object
trData = pModel.transform(power_ml_data)
trData

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, label: double, Month: bigint, Hour: double, Daytime: double, MonthVec: vector, PCAfeatures: vector, PCAresult: vector, features: vector]

In [25]:
#view transformed data
trData.show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|      MonthVec|         PCAfeatures|           PCAresult|            features|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|[1.79440486365695...|(17,[0,1,3,4,6],[...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|[1.80604083009823.

In [26]:
#view untruncated features column of transformed data
pModel.transform(power_ml_data).select("features").show(10, truncate = False)

+-------------------------------------------------------------------------------------+
|features                                                                             |
+-------------------------------------------------------------------------------------+
|(17,[0,1,3,4,6],[1.7944048636569536,-0.741274644740461,34055.6962,16128.87538,1.0])  |
|(17,[0,1,3,4,6],[1.8060408300982307,-0.7108534239558637,29814.68354,19375.07599,1.0])|
|(17,[0,1,3,4,6],[1.8102297630563897,-0.7283113191159094,29128.10127,19006.68693,1.0])|
|(17,[0,1,3,4,6],[1.7986676517408837,-0.7220913032200099,28228.86076,18361.09422,1.0])|
|(17,[0,1,3,4,6],[1.8632872016379705,-0.7323046647776716,27335.6962,17872.34043,1.0]) |
|(17,[0,1,3,4,6],[1.87820674500461,-0.7628199906979752,26624.81013,17416.41337,1.0])  |
|(17,[0,1,3,4,6],[1.915292987179555,-0.7636928919816571,25998.98734,16993.31307,1.0]) |
|(17,[0,1,3,4,6],[1.924005408070291,-0.7645388021423948,25446.07595,16661.39818,1.0]) |
|(17,[0,1,3,4,6],[1.895019303530

The transformed data produced by the pipeline looks the same as the transformed data generated above manually using the piepeline steps.

## Fit Elastic Net Linear Regression Model to Transformed Data

In [27]:
#define linear regression model
lr = LinearRegression(solver = "l-bfgs") #for large data sets

In [28]:
#set parameter grid for regularization and elastic net parameters
grid = ParamGridBuilder() \
  .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
  .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
  .build()

In [29]:
#define and view crossvalidator object
cv = CrossValidator(estimator = lr,
                    estimatorParamMaps = grid,
                    evaluator = RegressionEvaluator(metricName = "rmse"),
                    numFolds = 5,
                    parallelism = 128)
cv

CrossValidator_9b12d331e83f

In [30]:
#use the cross validator object to fit the model to the transformed data
cvModel = cv.fit(trData)

26/04/29 06:30:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [31]:
#view crossvalidator model object
cvModel

CrossValidatorModel_f645ec9e50cc

In [52]:
#save the model in current working directory
#cvModel.write().overwrite().save("p3Model")

In [41]:
#load the saved model from current working directory
#cvModel = CrossValidatorModel.load("p3Model")

In [32]:
#view CV error and corresponding tuning parameters of models fit, first few rows
my_list = []
for i in range(len(grid)):
    my_list.append([cvModel.avgMetrics[i], grid[i].values()])
print("RMSE, [regParam, elasticNetParam]")
my_list[:10]

RMSE, [regParam, elasticNetParam]


[[2148.122277489788, dict_values([0.0, 0.0])],
 [2148.1222774897883, dict_values([0.0, 0.05])],
 [2148.1222774897888, dict_values([0.0, 0.1])],
 [2148.1222774897888, dict_values([0.0, 0.25])],
 [2148.1222774897883, dict_values([0.0, 0.5])],
 [2148.1222774897888, dict_values([0.0, 0.75])],
 [2148.122277489788, dict_values([0.0, 0.9])],
 [2148.122277489788, dict_values([0.0, 0.95])],
 [2148.1222774897888, dict_values([0.0, 0.98])],
 [2148.1222774897888, dict_values([0.0, 0.99])]]

In [33]:
#extract tuning parameters for best fit, which are stored in cvModel
#find model fit with lowest error
comp = my_list[0][0]
for i in my_list:
    if i[0] < comp:
        comp = i[0]
        current_min = i
print("Average RMSE, Regularization Parameter, and Elastic Net Parameter for Best Fit CV Model:")
print([current_min])

Average RMSE, Regularization Parameter, and Elastic Net Parameter for Best Fit CV Model:
[[2148.1179531763105, dict_values([1.0, 0.05])]]


The regularization paramater controls the strength of regularizaiton for the model, and for this model it is 0.05. The elastic net parameter is a mixing parameter that balances between L1 and L2 regularization. In this model it is 0.1, more heavily weighted toward the L2 penalty. 

In [34]:
#Use model to create predictions on entire data set (training RSME)
preds = cvModel.transform(trData)
print(preds) #view data frame object
preds.show(5) #view first few rows

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, label: double, Month: bigint, Hour: double, Daytime: double, MonthVec: vector, PCAfeatures: vector, PCAresult: vector, features: vector, prediction: double]
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+--------------------+------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|      MonthVec|         PCAfeatures|           PCAresult|            features|        prediction|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+--------------------+------------------+
|   

In [35]:
#evaluate predictions with training RMSE
RegressionEvaluator().evaluate(preds)

2147.099998452766

The trianing RMSE is almost the same as the CV RMSE's above, but slightly better.

In [36]:
#residuals data frame with predictions, label, and residuals(label - prediction)
predsWithResDF = preds.select("label", "prediction") \
                .withColumn("residuals", col("label") - col("prediction"))
predsWithResDF.show(10)

+-----------+------------------+------------------+
|      label|        prediction|         residuals|
+-----------+------------------+------------------+
|20240.96386|20881.344727111693|-640.3808671116931|
|20131.08434|18658.123714108562|1472.9606258914391|
|19668.43373| 18202.63791612456|1465.7958138754402|
|18899.27711| 17588.65409901109|1310.6230109889075|
|18442.40964|16995.301370942656|1447.1082690573458|
|18130.12048|16515.736182965204|1614.3842970347978|
|17945.06024|16091.357427012863| 1853.702812987136|
|17459.27711| 15720.82693565721|  1738.45017434279|
|17025.54217| 15269.22700859221|1756.3151614077906|
|16794.21687|14936.541216817397|1857.6756531826031|
+-----------+------------------+------------------+
only showing top 10 rows


# Streaming Part

## Reading a Stream

A folder called csv_files is set up in the same directory as the .py file for creating the streaming data.

In [37]:
#set up schema for data stream
my_schema = StructType() \
                .add("Temperature", "double") \
                .add("Humidity", "double") \
                .add("Wind_Speed", "double") \
                .add("General_Diffuse_Flows", "double") \
                .add("Diffuse_Flows", "double") \
                .add("Power_Zone_1", "double") \
                .add("Power_Zone_2", "double") \
                .add("Power_Zone_3", "double") \
                .add("Month", "integer") \
                .add("Hour", "integer")

In [38]:
#create data frame to listen for streaming data coming into csv_files
input = spark.readStream.schema(my_schema).option("header", "true").csv("csv_files")
input

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, Power_Zone_3: double, Month: int, Hour: int]

## Transform/Aggregation Step

In [39]:
#use pipeline transformer to transform incoming data and model transformer to obtain predictions on the transformed data
#this is the first transformation/aggregation step for the streaming part

trInput1 = pModel.transform(input) #pipeline transformer
str_preds = cvModel.transform(trInput1) #cv model transformer
strPredsWithResDF = str_preds.select("label", "prediction") \
                        .withColumn("residuals", col("label") - col("prediction"))
strPredsWithResDF

DataFrame[label: double, prediction: double, residuals: double]

In [40]:
#rename response variable of incoming data to label
#second transformation
trInput2 = input.withColumnRenamed("Power_Zone_3", "label")
trInput2

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, label: double, Month: int, Hour: int]

In [41]:
#join the two input streams together on "label"column
joined_input = strPredsWithResDF.join(trInput2, "label", "inner")
joined_input

DataFrame[label: double, prediction: double, residuals: double, Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, Month: int, Hour: int]

## Writing Step

In [52]:
my_query = joined_input \
                .writeStream.outputMode("append") \
                .format("console") \
                .start()

26/04/29 07:58:04 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-fc9736e1-10e8-4232-b083-0f721b24d9d0. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/29 07:58:04 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15024.57831|16777.132184505877|-1752.5538745058766|      10.52|    74.6|     4.919|                0.048|        0.089| 34615.38462| 28554.54545|   11|  22|
|    18816.0|15493.073992415117| 3322.9260075848833|      20.13|   59.85|     0.076|                776.0|        125.7| 29686.63079| 15609.77597|    4|  14|
|11178.79518|10027.771147732918| 1151.0240322670816|      4.509|    74.5|     0.084|                6.643|       

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25432.25806|24769.467265103503|  662.7907948964967|      12.95|    80.6|     0.089|                0.769|        0.708| 42918.12766| 23667.07317|    3|  19|
|18699.63636| 18543.73006110062| 155.90629889937918|      19.17|    61.4|     0.073|                353.2|        352.1| 33691.96986|  18868.8391|    4|  17|
|24846.15075|24149.676539967288|   696.474210032713|       9.07|    81.4|     0.084|                0.037|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9640.336134| 7144.493633523194|  2495.842500476805|      12.64|    74.0|     0.078|                 0.07|        0.145| 21377.94677| 15718.93219|   12|   6|
|32113.80753|31207.561931425193|  906.2455985748056|      24.95|    76.4|      4.91|                0.051|         0.13| 39631.09635| 27618.98734|    7|  23|
|16903.71859|16810.010040727477|  93.70854927252367|      12.08|    86.3|     4.917|                285.8|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|         residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9825.542169| 9775.569259264823| 49.97290973517738|      15.31|    87.0|     0.074|                0.062|        0.107| 22295.38462| 16452.89256|   11|   5|
| 15648.3871|14664.410034981916| 983.9770650180835|      25.74|   35.97|     4.922|                571.1|        584.3| 31802.55319| 19445.12195|    3|  16|
|9922.689076|12698.692196241635|-2776.003120241634|      14.03|   68.81|     0.084|                39.11|        39.55

...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|         residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|21256.03239| 20228.71263736767| 1027.319752632331|      18.18|    87.5|     0.072|                0.062|        0.085| 35756.06557| 21321.36223|    5|  23|
|10967.27273|10256.335198245859| 710.9375317541417|      13.35|    88.4|     0.085|                146.9|        129.5|  19753.8859| 10173.11609|    4|   7|
| 18702.6506|17938.935576679538| 763.7150233204629|      20.08|    79.7|     0.066|                0.102|        0.04

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|         residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|31354.64435|29259.925634735442|2094.7187152645565|      30.66|   51.72|     0.067|                764.0|        140.0| 39803.32226| 27778.48101|    7|  15|
|12214.25945|12617.012197541655|-402.7527475416555|      19.97|   61.09|     0.275|                 0.08|        0.089|  27289.9115| 16192.51559|    9|   5|
|13915.96639|11992.099117319372|1923.8672726806271|      14.86|   64.12|     0.079|                25.07|        25.4

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|19422.69592| 21434.40204387016|-2011.7061238701608|       21.1|   56.56|     4.903|                 0.08|        0.104| 28819.17869|  19421.7529|    8|   3|
|28745.83072|26870.201483311557| 1875.6292366884445|      25.16|    85.7|     4.918|                0.088|        0.107| 36899.80022| 24447.30729|    8|   0|
|16145.45455|  17654.8159915194|-1509.3614415194006|      15.37|    78.4|     0.071|                144.6|      

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|19938.46154|19380.230274040794|  558.2312659592062|      23.93|   61.38|     0.057|                904.0|         42.4|  35399.2053| 22307.27651|    6|  13|
|24562.49246|25882.192371956746|-1319.6999119567445|      13.15|   55.61|     0.081|                 8.51|         8.46| 43932.20339|  25688.7538|    2|  19|
|14663.39698|14137.322205630655|  526.0747743693446|      15.56|   64.22|     0.077|                0.044|      

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|         residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|30179.74895|27067.890396145394|3111.8585538546067|      28.08|    71.4|     4.915|                440.5|        294.7| 36620.33223| 22841.77215|    7|  14|
|12984.80243| 13526.39192763584|-541.5894976358395|      21.93|   46.92|     4.951|                584.6|        38.93| 35694.35449| 21723.23651|   10|  10|
|9790.156062| 8822.795097113487| 967.3609648865131|      16.92|   59.98|     0.088|                 0.04|        0.09

In [53]:
my_query.stop()

26/04/29 08:09:04 WARN DAGScheduler: Failed to cancel job group 187ce139-e018-4bec-b079-b0ac2674e997. Cannot find active jobs for it.
26/04/29 08:09:05 WARN DAGScheduler: Failed to cancel job group 187ce139-e018-4bec-b079-b0ac2674e997. Cannot find active jobs for it.


## Producing Data

In [102]:
#import modules for producing data
#import time
#import pandas as pd
#import numpy as np

In [81]:
#testing code for .py file here
#create loop for outputting data to csv files to be read as streaming data
#randomly sample five rows of data at a time to write to csv files
#put in a pause between loops

#import time
#pwr_str_data = pd.read_csv("power_streaming_data.csv")

#for i in range(0, 20):
#    temp = pwr_str_data.loc[np.random.randint(pwr_str_data.shape[0], size = 5)]
#    temp.to_csv("csv_files/pwr_str_data" + str(i) + ".csv", index = False, header = True)
#    time.sleep(15)